In [45]:
import pandas as pd
import numpy as np
import os

# Cargar datos
ruta = os.path.join("..", 'Data', 'TouristAccommodationRaw02022026.csv')

df = pd.read_csv(ruta, encoding='latin1')

## 1. Columnas para multiples departamentos

En esta fase, preparamos unas columnas que facilitan el analisis de varios departamentos. El objetivo principal es normalizar las categorías y asegurar que las métricas sean columnas númericas sin reducir el tamaño de la muestra (10000 registros).

* **Índice unico** Se crea una columna `unique_id` para identificar el id unico de todos los registros.
* **Normalización de Texto:** Se eliminan espacios en blanco en las columnas `city` y `room_type` para evitar duplicidad de categorías.
* **Consistencia de Tipos:** Se fuerza el tipo a `float` en las columnas de reseñas para permitir cálculos estadísticos precisos.

In [46]:
# (1) Crear una columna de indice para identificar el id unico de registros

# Este columna de indice 'unique_id' con valor empieza desde 1 hasta el numero de todos registros
df['unique_id'] = np.arange(1, len(df)+1)

In [47]:
# (2) NORMALIZACIÓN DE CATEGORÍAS

# Eliminamos espacios en blanco, estandarizamos a minúsculas y capitalizamos
# (ej. "madrid " -> "Madrid")
df['city'] = df['city'].str.strip().str.capitalize()


# Limpiamos los tipos de alojamiento
df['room_type'] = df['room_type'].str.strip()

In [48]:
# --- 3. IMPUTACIÓN DE PRECIOS ---

# Paso A: Imputación segmentada (Por Ciudad y Tipo de Habitación)
df['price'] = df.groupby(['city', 'room_type'])['price'].transform(
    lambda x: x.fillna(x.median())
)

# Paso B: Imputación de seguridad
# En caso de que un segmento completo sea nulo, usamos la mediana de todo el dataset
df['price'] = df['price'].fillna(df['price'].median())

# --- 4. AJUSTE DE TIPOS ---
# Aseguramos que price sea numérico para el análisis comercial
df['price'] = df['price'].astype(float)

In [49]:
# (5) Columnas de reseñas por clientes, para analisis de Marketing y Experiencia de Clientes

# Definición de las columnas de reseñas
columnas_rating = [
    'review_scores_rating',        # Evaluación general
    'review_scores_accuracy',      # Precisión de detalles
    'review_scores_cleanliness',   # Higiene
    'review_scores_checkin',       # Proceso de entrada
    'review_scores_communication', # Comunicación
    'review_scores_location',      # Zona
    'review_scores_value'          # Valor
]

# Aseguramos el tipo numérico. Los errores o celdas vacías se convierten en NaN.
for col in columnas_rating:
    df[col] = pd.to_numeric(df[col], errors='coerce')

## 2. Limpieza y Preparación: Marketing y Estrategia Comercial

En esta fase, preparamos los datos para el análisis de mercado. El objetivo principal es normalizar las categorías y asegurar que las métricas sean columnas númericas sin reducir el tamaño de la muestra (10000 registros).

* **Normalización de Texto:** Se modifica la columna `neighbourhood_name` para los barrios con nombres codificados correctamente, y se crea una column `city_neighbourhood` que lleva nombre de la ciudad y del barrio.
* **Consistencia de Tipos:** Se fuerza el tipo a `float` en las columnas `minimum_nights` y `maximum_nights` para permitir cálculos estadísticos precisos.

In [50]:
# (1) Limpieza para los nombres de los barrios
# Muchos barrios tienen los nombres mal codificados en la base de datos aunque usamos "encoding =" con cualquier método

# --Paso A:-- Creamos un diccionario para guardar los nombres corrctos de los mal codificados,
# con la ayuda de IA identificando todos los valores únicos de "neighbourhood_name".
# Aqui los signos falsos son del metodo encoding='latin1'
clean_neighbourhood = {
    # Barcelona (nombres en catalán)
    'Can Barï¿½': 'Can Baró',
    'Diagonal Mar i el Front Marï¿½tim del Poblenou': 'Diagonal Mar i el Front Marítim del Poblenou',
    'Provenï¿½als del Poblenou': 'Provençals del Poblenou',
    'Sant Genï¿½s dels Agudells': 'Sant Genís dels Agudells',
    'Sant Martï¿½ de Provenï¿½als': 'Sant Martí de Provençals',
    'Sant Pere, Santa Caterina i la Ribera': 'Sant Pere, Santa Caterina i la Ribera',
    'Sarriï¿½': 'Sarrià',
    'Torre Barï¿½': 'Torre Baró',
    'Vilapicina i la Torre Llobeta': 'Vilapicina i la Torre Llobeta',
    'el Baix Guinardï¿½': 'el Baix Guinardó',
    'el Barri Gï¿½tic': 'el Barri Gòtic',
    'el Besï¿½s i el Maresme': 'el Besòs i el Maresme',
    "el Camp d'en Grassot i Grï¿½cia Nova": "el Camp d'en Grassot i Gràcia Nova",
    'el Congrï¿½s i els Indians': 'el Congrés i els Indians',
    'el Guinardï¿½': 'el Guinardó',
    'el Putxet i el Farrï¿½': 'el Putxet i el Farró',
    'el Turï¿½ de la Peira': 'el Turó de la Peira',
    'la Sagrada Famï¿½lia': 'la Sagrada Família',
    'la Vila Olï¿½mpica del Poblenou': 'la Vila Olímpica del Poblenou',
    'la Vila de Grï¿½cia': 'la Vila de Gràcia',

    # Girona & Mallorca & Menorca (nombres en catalán)
    'Arbï¿½cies': 'Arbúcies',
    'Besalï¿½': 'Besalú',
    'Bescanï¿½': 'Bescanó',
    'Bï¿½scara': 'Bàscara',
    'Cadaquï¿½s': 'Cadaqués',
    'Campdevï¿½nol': 'Campdevànol',
    "Bellcaire d'Empordï¿½": "Bellcaire d'Empordà",
    "Castellï¿½ d'Empï¿½ries": "Castelló d'Empúries",
    'Celrï¿½': 'Celrà',
    'Cornellï¿½ del Terri': 'Cornellà del Terri',
    "Cruï¿½lles, Monells i Sant Sadurnï¿½ de l'Heura": "Cruïlles, Monells i Sant Sadurní de l'Heura",
    'Foixï¿½': 'Foixà',
    'Isï¿½vol': 'Isòvol',
    'Lladï¿½': 'Lladó',
    'Llanï¿½ï¿½': 'Llançà',
    'Llï¿½via': 'Llívia',
    'Maiï¿½ de Montcal': 'Maià de Montcal',
    'Maï¿½anet de la Selva': 'Maçanet de la Selva',
    'Palamï¿½s': 'Palamós',
    'Palau de Santa Eulï¿½lia': 'Palau de Santa Eulàlia',
    'Parlavï¿½': 'Parlavà',
    'Pontï¿½s': 'Pontós',
    'Puigcerdï¿½': 'Puigcerdà',
    'Rabï¿½s': 'Rabós',
    'Regencï¿½s': 'Regencós',
    'Sant Feliu de Guï¿½xols': 'Sant Feliu de Guíxols',
    'Sant Juliï¿½ de Ramis': 'Sant Julià de Ramis',
    'Sant Llorenï¿½ de la Muga': 'Sant Llorenç de la Muga',
    'Sant Martï¿½ de Llï¿½mena': 'Sant Martí de Llémena',
    'Sant Miquel de Fluviï¿½': 'Sant Miquel de Fluvià',
    'Sant Pau de Segï¿½ries': 'Sant Pau de Segúries',
    'Serinyï¿½': 'Serinyà',
    'Torroella de Montgrï¿½': 'Torroella de Montgrí',
    'Tortellï¿½': 'Tortellà',
    'Ullï¿½': 'Ullà',
    'Urï¿½s': 'Urús',
    'Vallfogona de Ripollï¿½s': 'Vallfogona de Ripollès',
    'Ventallï¿½': 'Ventalló',
    'Vidrï¿½': 'Vidrà',
    'Vilajuï¿½ga': 'Vilajuïga',
    'Vilaï¿½r': 'Vilaür',
    'Alarï¿½': 'Alaró',
    'Alcï¿½dia': 'Alcúdia',
    'Artï¿½': 'Artà',
    'Bï¿½ger': 'Búger',
    'Calviï¿½': 'Calvià',
    'Deyï¿½': 'Deyá',
    'Llubï¿½': 'Llubí',
    'Marratxï¿½': 'Marratxí',
    'Montuï¿½ri': 'Montuïri',
    'Pollenï¿½a': 'Pollença',
    'Sant Llorenï¿½ des Cardassar': 'Sant Llorenç des Cardassar',
    'Santa Eugï¿½nia': 'Santa Eugènia',
    'Santa Marï¿½a del Camï¿½': 'Santa María del Camí',
    'Santanyï¿½': 'Santanyí',
    'Sï¿½ller': 'Sóller',
    'Mahï¿½n': 'Mahón',
    'Sant Lluï¿½s': 'Sant Lluís',
    'Torroella de Fluviï¿½': 'Torroella de Fluvià',
    'Sant Martï¿½ Vell': 'Sant Martí Vell',
    "La Tallada d'Empordï¿½": "La Tallada d'Empordà",
    'Rupiï¿½': 'Rupià',

    # Madrid & Sevilla (nombres en español)
    'Argï¿½elles': 'Argüelles',
    'Casco Histï¿½rico de Barajas': 'Casco Histórico de Barajas',
    'Casco Histï¿½rico de Vallecas': 'Casco Histórico de Vallecas',
    'Casco Histï¿½rico de Vicï¿½lvaro': 'Casco Histórico de Vicálvaro',
    'Concepciï¿½n': 'Concepción',
    'Cï¿½rmenes': 'Cármenes',
    'El Plantï¿½o': 'El Plantío',
    'Entrevï¿½as': 'Entrevías',
    'Fontarrï¿½n': 'Fontarrón',
    'Hellï¿½n': 'Hellín',
    'Hispanoamï¿½rica': 'Hispanoamérica',
    'Jerï¿½nimos': 'Jerónimos',
    'Moscardï¿½': 'Moscardó',
    'Niï¿½o Jesï¿½s': 'Niño Jesús',
    'Nueva Espaï¿½a': 'Nueva España',
    'Opaï¿½el': 'Opañel',
    'Pacï¿½fico': 'Pacífico',
    'Peï¿½agrande': 'Peñagrande',
    'San Andrï¿½s': 'San Andrés',
    'Timï¿½n': 'Timón',
    'Barrio Leï¿½n': 'Barrio León',
    'Carretera de Carmona, Marï¿½a Auxiliadora, Fontanal': 'Carretera de Carmona, María Auxiliadora, Fontanal',
    'Ciudad Jardï¿½n': 'Ciudad Jardín',
    'Doctor Barraquer, G. Renfe, Policlï¿½nico': 'Doctor Barraquer, G. Renfe, Policlínico',
    'El Rocï¿½o': 'El Rocío',
    'El Tardï¿½n, El Carmen': 'El Tardón, El Carmen',
    'Encarnaciï¿½n, Regina': 'Encarnación, Regina',
    'Heliï¿½polis': 'Heliópolis',
    'La Palmilla, Doctor Maraï¿½ï¿½n': 'La Palmilla, Doctor Marañón',
    'Leï¿½n XIII, Los Naranjos': 'León XIII, Los Naranjos',
    'Nerviï¿½n': 'Nervión',
    'Prado, Parque Marï¿½a Luisa': 'Prado, Parque María Luisa',
    'San Bartolomï¿½': 'San Bartolomé',
    'San Josï¿½ Obrero': 'San José Obrero',
    'Tiro de Lï¿½nea, Santa Genoveva': 'Tiro de Línea, Santa Genoveva',
    'San Juliï¿½n': 'San Julián',
    'San Fermï¿½n': 'San Fermín',
    'Zofï¿½o': 'Zofío'
}

# --Paso B:-- Sustuimso los nombres mal codificados por los correctos
df['neighbourhood_name'] = df['neighbourhood_name'].replace(clean_neighbourhood)


# --Paso C:-- Verificamos si todavía quedan barrios mal con nombres mal codificados
remaining_bad = df[df['neighbourhood_name'].str.contains('ï¿½', na=False)]['neighbourhood_name'].unique()
if len(remaining_bad) == 0:
    print("EXITO: Todos errores identificados de encoding se han limpiado.")
else:
    print(f"Quedan problemáticos: {remaining_bad}")


# --Paso D:-- Ponemos los nombres de barrios valencianos en forma de título
df['neighbourhood_name'] = df['neighbourhood_name'].str.title()

EXITO: Todos errores identificados de encoding se han limpiado.


In [51]:
# (2) Creamos una columna que lleva ambos nombres de la ciudad y del barrio
df['city_neighbourhood'] = df['city'] + ": " + df['neighbourhood_name']

In [52]:
# (3) Columns de disponibilidad mínima y máxima de noches

# Asegurar el tipo numérico
df['minimum_nights'] = pd.to_numeric(df['minimum_nights'], errors='coerce')
df['maximum_nights'] = pd.to_numeric(df['maximum_nights'], errors='coerce')

## 3. Limpieza para Experiencia del Cliente.

Para responder a las preguntas sobre satisfacción sin reducir la muestra total de 10000 registros, se aplica la siguiente lógica:

* **Tratamiento de catidad de reseñas:** Se transforman a formato numérico, manteniendo los valores ausentes como `NaN`.

In [53]:
#Columns de cantidad de reseñas

# Asegurar el tipo numérico
df['number_of_reviews'] = pd.to_numeric(df['number_of_reviews'], errors='coerce')
df['reviews_per_month'] = pd.to_numeric(df['reviews_per_month'], errors='coerce')

## 4. Limpieza para Operaciones y Gestión de Inventario.

In [54]:
# Asegurar el tipo numérico
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce')
df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
df['beds'] = pd.to_numeric(df['beds'], errors='coerce')

IMPUTACIÓN DE VALORES NULOS Y CEROS

BEDROOMS
Estrategia:

Para Private room, Shared room y Hotel room → se imputa como 1 dormitorio/
Para Entire home/apt → se utiliza la mediana de dormitorios según room_type y accommodates

In [55]:
# Referencia: solo valores válidos de bedrooms (>0)
bedrooms_ref = df[df['bedrooms'] > 0]

# Mediana de bedrooms por tipo de habitación y capacidad
bedrooms_median = bedrooms_ref.groupby(
    ['room_type', 'accommodates']
)['bedrooms'].median()

def impute_bedrooms(row):
    if pd.isna(row['bedrooms']) or row['bedrooms'] == 0:
        if row['room_type'] in ['Private room', 'Shared room', 'Hotel room']:
            return 1
        key = (row['room_type'], row['accommodates'])
        if key in bedrooms_median:
            return bedrooms_median[key]
        return 1
    return row['bedrooms']

df['bedrooms'] = df.apply(impute_bedrooms, axis=1)

BATHROOMS
Estrategia:

Se imputan valores nulos o iguales a 0 utilizando la mediana por room_type y bedrooms
En caso de no existir grupo, se usa la mediana global

In [56]:
# Mediana de bathrooms por tipo de habitación y dormitorios
bathroom_median = df[df['bathrooms'] > 0].groupby(
    ['room_type', 'bedrooms']
)['bathrooms'].median()

def impute_bathrooms(row):
    if pd.isna(row['bathrooms']) or row['bathrooms'] == 0:
        key = (row['room_type'], row['bedrooms'])
        if key in bathroom_median:
            return bathroom_median[key]
        return df['bathrooms'].median()
    return row['bathrooms']

df['bathrooms'] = df.apply(impute_bathrooms, axis=1)

 BEDS
Estrategia:

Se utiliza la mediana por room_type y accommodates
Se asegura coherencia lógica con el número de dormitorios

In [57]:
# Mediana de beds por tipo de habitación y capacidad
beds_median = df[df['beds'] > 0].groupby(
    ['room_type', 'accommodates']
)['beds'].median()

def impute_beds(row):
    if pd.isna(row['beds']) or row['beds'] == 0:
        key = (row['room_type'], row['accommodates'])
        if key in beds_median:
            return max(row['bedrooms'], beds_median[key])
        return max(row['bedrooms'], 1)
    return row['beds']

df['beds'] = df.apply(impute_beds, axis=1)

RESTRICCIONES LÓGICAS

In [58]:
# Valores mínimos razonables
df['bedrooms'] = df['bedrooms'].clip(lower=1)
df['beds'] = df['beds'].clip(lower=1)
df['bathrooms'] = df['bathrooms'].clip(lower=0.5)

# Coherencia: el número de camas no puede ser menor que el número de dormitorios
df.loc[df['beds'] < df['bedrooms'], 'beds'] = df['bedrooms']

CONVERSIÓN DE TIPOS DE DATOS

In [59]:
df['bedrooms'] = df['bedrooms'].round().astype(int)
df['beds'] = df['beds'].round().astype(int)
df['bathrooms'] = df['bathrooms'].astype(float)

In [60]:
#Crear una columna de ocupacion_30(en dias)
df["ocupacion_30"] = 30 - df["availability_30"]

## 4. Limpieza para KPIs

In [61]:
#COMPROBAR AVAILABILITY

df_hav = df[
    (df['availability_30'] > df['availability_60']) &
    (df['availability_60'] > df['availability_90']) &
    (df['availability_90'] > df['availability_365'])
].copy()

df_hav


,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,unique_id,city_neighbourhood,ocupacion_30


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 38 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 10000 non-null  int64  
 1   name                         9997 non-null   object 
 2   description                  9862 non-null   object 
 3   host_id                      10000 non-null  int64  
 4   neighbourhood_name           10000 non-null  object 
 5   neighbourhood_district       6079 non-null   object 
 6   room_type                    10000 non-null  object 
 7   accommodates                 10000 non-null  int64  
 8   bathrooms                    10000 non-null  float64
 9   bedrooms                     10000 non-null  int64  
 10  beds                         10000 non-null  int64  
 11  amenities_list               9983 non-null   object 
 12  price                        10000 non-null  float64
 13  minimum_nights   

## 6. Exportación de Resultados
Una vez finalizado el proceso de limpieza y normalización para los tres departamentos se procede a exportar el Dataset Limpio.

* **Ruta de destino:** Se almacena en la carpeta institucional /Data/ bajo el nombre TouristAccommodationClean19012026.csv.
* **Codificación:** Se utiliza `latin1` para garantizar que la corrección de caracteres especiales.

In [ ]:
# --- EXPORTACIÓN DEL DATASET LIMPIO ---

# 1. Nombre del nuevo archivo
nombre_archivo_limpio = 'TouristAccommodationClean02022026.csv'

# 2. Construimos la ruta apuntando a la misma carpeta 'Data'
# Usamos '..' para subir un nivel y luego entrar en 'Data'
ruta_guardado = os.path.join('..', 'Data', nombre_archivo_limpio)

# 3. Guardamos el DataFrame
# index=False evita que se cree una columna extra de números

# encoding='utf-8' para evitar errores de leer el dataframe,
# porque pd.read_csv() va a leer con "encoding = 'utf-8'" automaticamente y aparecería errores con "latin1"
df.to_csv(ruta_guardado, index=False, encoding='utf-8')